# Exploration of Sample Stream (NHDPlus) Dataset

In [1]:
from pathlib import Path                                                                                                                                                      
import sys                                                                                                                                                                    
import pandas as pd                                                                                                                                                           
import geopandas as gpd                                                                                                                                                       
                                                                                                                                                                            
ROOT = Path("/courses/DS5500.202710/teams/team4")                                                                                                                             
sys.path.insert(0, str(ROOT))                                                                                                                                                 
from etl.common.nhdplus import SLIM_LAYERS, find_huc8_geodatabase, is_hu4_fallback                                                                                            
                                                                                                                                                                            
EXTRACT_DIR = ROOT / "data/processed/nhdplus_extracted"                                                                                                                       
packages = sorted(EXTRACT_DIR.glob("*/*.gpkg"))                                                                                                                               
                                                                                                                                                                            
sizes_mb = [p.stat().st_size / 1e6 for p in packages]                                                                                                                         
print(f"Found {len(packages)} extracted packages ({sum(sizes_mb) / 1e3:.1f} GB)")                                                                                             
print(f"Each keeps only: {', '.join(SLIM_LAYERS)}\n")                                                                                                                         
for p, mb in list(zip(packages, sizes_mb))[:10]:                                                                                                                              
  print(f"{p.parent.name}: {mb:.1f} MB")

Found 152 extracted packages (88.6 GB)
Each keeps only: NHDFlowline, NHDPlusFlow, NHDPlusCatchment, NHDPlusFlowlineVAA

NHDPLUS_H_0101_HU4_20220901_GDB: 292.6 MB
NHDPLUS_H_0102_HU4_20230713_GDB: 229.1 MB
NHDPLUS_H_0103_HU4_20230713_GDB: 243.3 MB
NHDPLUS_H_0104_HU4_20230713_GDB: 98.3 MB
NHDPLUS_H_0105_HU4_20230713_GDB: 210.3 MB
NHDPLUS_H_0106_HU4_20230713_GDB: 157.7 MB
NHDPLUS_H_0107_HU4_20230713_GDB: 184.8 MB
NHDPLUS_H_0108_HU4_20220324_GDB: 493.2 MB
NHDPLUS_H_0109_HU4_20230713_GDB: 213.0 MB
NHDPLUS_H_0110_HU4_20220323_GDB: 265.3 MB


## Pick one extracted package for inspection

In [2]:
huc8 = "01080205"   # change to any HUC8 in your case study                                                                                                                   
                                                                                                                                                                                
gdb_path = find_huc8_geodatabase(EXTRACT_DIR, huc8)                                                                                                                           
print("Package:", gdb_path)                                                                                                                                                   
if is_hu4_fallback(gdb_path, huc8):                                                                                                                                           
  print(f"Note: no HU8 package for {huc8}; this is the HU4 package covering it")

Package: /courses/DS5500.202710/teams/team4/data/processed/nhdplus_extracted/NHDPLUS_H_0108_HU4_20220324_GDB/NHDPLUS_H_0108_HU4_20220324_GDB.gpkg
Note: no HU8 package for 01080205; this is the HU4 package covering it


## List all layers in GDB

In [3]:
import geopandas as gpd
layers = gpd.list_layers(gdb_path)                                                                                                                                            
layers.columns = ["layer_name", "geometry_type"]                                                                                                                              
display(layers)

,layer_name,geometry_type
0,NHDFlowline,MultiLineString Z
1,NHDPlusCatchment,MultiPolygon
2,NHDPlusFlow,NaN
3,NHDPlusFlowlineVAA,NaN


## List features for NHDFlowline layer

In [4]:
stream_layer = "NHDFlowline"

streams_sample = gpd.read_file(
    gdb_path,
    layer=stream_layer,
    rows=5,
)

print("CRS:", streams_sample.crs)
print("Geometry types:", streams_sample.geom_type.unique())
print("Sample dimensions:", streams_sample.shape)

print("\nColumns:")
print(streams_sample.columns.tolist())

display(streams_sample.drop(columns="geometry", errors="ignore").T)

CRS: EPSG:5498
Geometry types: <ArrowStringArray>
['MultiLineString']
Length: 1, dtype: str
Sample dimensions: (5, 19)

Columns:
['permanent_identifier', 'fdate', 'resolution', 'gnis_id', 'gnis_name', 'lengthkm', 'reachcode', 'flowdir', 'wbarea_permanent_identifier', 'ftype', 'fcode', 'mainpath', 'innetwork', 'visibilityfilter', 'nhdplusid', 'vpuid', 'Shape_Length', 'Enabled', 'geometry']


,0,1,2,3,4
permanent_identifier,123020046,123020039,123019220,123019162,123021378
fdate,2012-03-07 21:30:32+00:00,2012-03-07 21:30:32+00:00,2012-03-07 21:28:33+00:00,2012-03-07 21:30:44+00:00,2015-02-25 00:00:00+00:00
resolution,2,2,2,2,2
gnis_id,None,None,None,None,None
gnis_name,None,None,None,None,None
lengthkm,0.014,0.051,0.11,0.049,0.540822
reachcode,01080205000976,01080205000976,01080205013787,01080205013736,01080205017617
flowdir,1,1,0,0,1
wbarea_permanent_identifier,141864230,141864230,NaN,NaN,141864230
ftype,558,558,336,336,558


In [5]:
# Examine feature types, identifiers, and missing values
attributes = streams_sample.drop(columns="geometry", errors="ignore")

summary = pd.DataFrame({
    "dtype": attributes.dtypes.astype(str),
    "non_null_sample": attributes.notna().sum(),
    "unique_sample": attributes.nunique(dropna=True),
})

display(summary)

,dtype,non_null_sample,unique_sample
permanent_identifier,str,5,5
fdate,"datetime64[ms, UTC]",5,4
resolution,int32,5,1
gnis_id,object,0,0
gnis_name,object,0,0
lengthkm,float64,5,5
reachcode,str,5,4
flowdir,int32,5,2
wbarea_permanent_identifier,str,3,1
ftype,int32,5,2


## Show stream layer with respective watershed, dam removal(s), and water quality station(s):

In [8]:
ROOT = Path("/courses/DS5500.202710/teams/team4")
huc4 = "0108"
EXTRACT_DIR = ROOT / "data/processed/nhdplus_extracted"

station_file = ROOT / "data/raw/wqp/results/pilot_station_selection.csv"
dam_file = next(ROOT.rglob("ARDamRemovalList_Figshare_Mar2026.csv"))
wbd_path = next(ROOT.rglob("WBD_National_GPKG.gpkg"))

# Load HUC8 watershed boundaries.
wbd_layers = gpd.list_layers(wbd_path)
hu8_layer = next(
    name for name in wbd_layers["name"]
    if "hu8" in name.lower()
)

watersheds = gpd.read_file(wbd_path, layer=hu8_layer)

huc_col = next(
    c for c in watersheds.columns if c.lower() == "huc8"
)

watersheds[huc_col] = (
    watersheds[huc_col].astype(str).str.strip().str.zfill(8)
)

# Select every HUC8 within the pilot HUC4.
watersheds_huc4 = watersheds[
    watersheds[huc_col].str.startswith(huc4)
].to_crs("EPSG:4326").copy()

if watersheds_huc4.empty:
    raise ValueError(f"No HUC8 watersheds found for HUC4 {huc4}")

huc8_codes = sorted(watersheds_huc4[huc_col].unique())

print(f"HUC4: {huc4}")
print(f"Number of HUC8 watersheds: {len(huc8_codes)}")
print("HUC8 codes:", huc8_codes)

HUC4: 0108
Number of HUC8 watersheds: 14
HUC8 codes: ['01080101', '01080102', '01080103', '01080104', '01080105', '01080106', '01080107', '01080201', '01080202', '01080203', '01080204', '01080205', '01080206', '01080207']


In [9]:
# Load all WQP station files.
WQP_DIR = ROOT / "data" / "raw" / "wqp"
STATION_DIR = WQP_DIR / "stations"

station_files = sorted(STATION_DIR.glob("*.csv"))
frames = []

for f in station_files:
    df = pd.read_csv(f, low_memory=False)
    df["source_huc8"] = f.stem
    frames.append(df)

if not frames:
    raise ValueError(f"No station CSVs found in {STATION_DIR}")

stations = pd.concat(frames, ignore_index=True)

# Keep surface-water stations.
surface_water_types = ["Stream", "River/Stream"]

surface_stations = stations[
    stations["MonitoringLocationTypeName"].isin(surface_water_types)
].copy()

# Validate coordinates and remove duplicate stations.
stations_map = surface_stations.copy()

stations_map["LatitudeMeasure"] = pd.to_numeric(
    stations_map["LatitudeMeasure"], errors="coerce"
)
stations_map["LongitudeMeasure"] = pd.to_numeric(
    stations_map["LongitudeMeasure"], errors="coerce"
)

stations_map = stations_map[
    stations_map["LatitudeMeasure"].between(-90, 90)
    & stations_map["LongitudeMeasure"].between(-180, 180)
].copy()

stations_map = stations_map.drop_duplicates(
    subset="MonitoringLocationIdentifier"
)

stations_gdf = gpd.GeoDataFrame(
    stations_map,
    geometry=gpd.points_from_xy(
        stations_map["LongitudeMeasure"],
        stations_map["LatitudeMeasure"],
    ),
    crs="EPSG:4326",
)

# Select stations geographically inside any HUC8 in the pilot HUC4.
stations_huc8 = gpd.sjoin(
    stations_gdf,
    watersheds_huc4[["geometry"]],
    how="inner",
    predicate="within",
).drop(columns="index_right", errors="ignore")

print(f"Total surface-water stations: {len(stations_map):,}")
print(f"Stations inside HUC4 {huc4}: {len(stations_huc8):,}")

Total surface-water stations: 347,370
Stations inside HUC4 0108: 8,412


## We can now use the HUC8s of the water quality stations to filter out the features in different HUC8s

In [10]:
# Count stations in each HUC8.
station_counts = gpd.sjoin(
    stations_gdf,
    watersheds_huc4[[huc_col, "geometry"]],
    how="inner",
    predicate="within",
).groupby(huc_col).size().sort_values(ascending=False)

print(station_counts)

huc8
01080205    1771
01080207    1072
01080106     821
01080201     803
01080203     576
01080107     569
01080204     478
01080206     452
01080103     414
01080101     374
01080104     351
01080202     308
01080105     254
01080102     169
dtype: int64


In [11]:
pilot_huc8 = station_counts.index[0]
print("Pilot HUC8:", pilot_huc8)

pilot_watershed = watersheds_huc4[
    watersheds_huc4[huc_col] == pilot_huc8
].copy()

Pilot HUC8: 01080205


In [12]:
dams = pd.read_csv(dam_file, low_memory=False)

dam_lat_col = "Latitude"
dam_lon_col = "Longitude"
dam_name_col = "Dam Name"

dams[dam_lat_col] = pd.to_numeric(
    dams[dam_lat_col], errors="coerce"
)
dams[dam_lon_col] = pd.to_numeric(
    dams[dam_lon_col], errors="coerce"
)

dams_valid = dams.dropna(
    subset=[dam_lat_col, dam_lon_col]
).copy()

dams_geo = gpd.GeoDataFrame(
    dams_valid,
    geometry=gpd.points_from_xy(
        dams_valid[dam_lon_col],
        dams_valid[dam_lat_col],
    ),
    crs="EPSG:4326",
)

dams_huc8 = gpd.sjoin(
    dams_geo,
    watersheds_huc4[["geometry"]],
    how="inner",
    predicate="within",
).drop(columns="index_right", errors="ignore")

print(f"Dam removals inside HUC4 {huc4}: {len(dams_huc8):,}")

Dam removals inside HUC4 0108: 82


In [13]:
stations_pilot = gpd.sjoin(
    stations_gdf,
    pilot_watershed[[huc_col, "geometry"]],
    how="inner",
    predicate="within",
).drop(columns="index_right", errors="ignore")

dams_pilot = gpd.sjoin(
    dams_geo,
    pilot_watershed[[huc_col, "geometry"]],
    how="inner",
    predicate="within",
).drop(columns="index_right", errors="ignore")

print(f"Stations in pilot HUC8: {len(stations_pilot):,}")
print(f"Dam removals in pilot HUC8: {len(dams_pilot):,}")

Stations in pilot HUC8: 1,771
Dam removals in pilot HUC8: 9


In [14]:
print("Preparing pilot HUC8...", flush=True)

# Project the pilot watershed.
pilot_watershed_projected = pilot_watershed.to_crs("EPSG:5070")

# Load NHDPlus flowlines from the HUC4 geodatabase.
print("Loading NHDPlus flowlines...", flush=True)
streams = gpd.read_file(gdb_path, layer=stream_layer)

print(f"Loaded {len(streams):,} flowlines.", flush=True)

# Project flowlines to a meter-based CRS.
print("Projecting streams...", flush=True)
streams = streams.to_crs("EPSG:5070")

# Use the spatial index to find flowlines intersecting the pilot HUC8.
print("Finding flowlines within pilot area...", flush=True)
pilot_geometry = pilot_watershed_projected.geometry.union_all()

candidate_idx = streams.sindex.query(
    pilot_geometry,
    predicate="intersects",
)

streams_pilot = streams.iloc[candidate_idx].copy()

print(f"Candidate flowlines: {len(streams_pilot):,}", flush=True)

# Clip candidate flowlines to the pilot HUC8 boundary.
print("Clipping candidate flowlines...", flush=True)
streams_pilot = gpd.clip(
    streams_pilot,
    pilot_watershed_projected,
)

print(f"Pilot HUC8 stream features: {len(streams_pilot):,}", flush=True)

# Simplify a separate copy for mapping only.
streams_map = streams_pilot[["geometry"]].copy()

streams_map["geometry"] = streams_map.geometry.simplify(
    tolerance=100,
    preserve_topology=False,
)

streams_map = streams_map.to_crs("EPSG:4326")

# Prepare the other map layers using the same pilot HUC8.
watersheds_map = pilot_watershed.to_crs("EPSG:4326")
stations_map = stations_pilot.to_crs("EPSG:4326")
dams_map = dams_pilot.to_crs("EPSG:4326")

print("All pilot layers prepared.", flush=True)
print(f"Watersheds: {len(watersheds_map):,}")
print(f"Streams: {len(streams_map):,}")
print(f"Stations: {len(stations_map):,}")
print(f"Dam removals: {len(dams_map):,}")

Preparing pilot HUC8...
Loading NHDPlus flowlines...
Loaded 104,740 flowlines.
Projecting streams...
Finding flowlines within pilot area...
Candidate flowlines: 13,538
Clipping candidate flowlines...
Pilot HUC8 stream features: 13,538
All pilot layers prepared.
Watersheds: 1
Streams: 13,538
Stations: 1,771
Dam removals: 9


In [15]:
import folium

# Center the map on the pilot HUC8.
minx, miny, maxx, maxy = watersheds_map.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(
    location=center,
    zoom_start=9,
    tiles="OpenStreetMap",
)

# Pilot HUC8 boundary.
folium.GeoJson(
    watersheds_map[[huc_col, "geometry"]].to_json(),
    name="Pilot HUC8 watershed",
    style_function=lambda feature: {
        "color": "navy",
        "weight": 2,
        "fillOpacity": 0.05,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[huc_col],
        aliases=["HUC8:"],
    ),
).add_to(m)

# Filtered, simplified stream reaches.
folium.GeoJson(
    streams_map.to_json(),
    name="NHDPlus streams",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 1,
    },
).add_to(m)

# Water-quality stations.
station_group = folium.FeatureGroup(name="Water-quality stations")

for _, row in stations_map.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.6,
        tooltip=str(row["MonitoringLocationIdentifier"]),
    ).add_to(station_group)

station_group.add_to(m)

# Dam removals within the pilot HUC8.
dam_group = folium.FeatureGroup(name="Dam removals")

for _, row in dams_map.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=str(row.get(dam_name_col, "Unknown dam")),
        icon=folium.Icon(color="darkred", icon="tint"),
    ).add_to(dam_group)

dam_group.add_to(m)

folium.LayerControl().add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])

print("Map ready.", flush=True)
m

Map ready.
